# Phase 2 — Azure Setup

**Run once per project.** Registers compute, environment, and data asset in Azure ML.

Prerequisites:
- Phase 1 gate passed (`data/processed/train_clean.csv` exists)
- `config.json` filled with your subscription / resource group / workspace
- Azure asset names filled in README_TEMPLATE.md → paste them in the CONFIG cell below

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
# Change PROJECT_NAME. Everything else derives from it.
PROJECT_NAME    = 'my-project'

DATA_ASSET_NAME = f'{PROJECT_NAME}-data'
COMPUTE_NAME    = 'aml-cluster'
ENV_NAME        = f'{PROJECT_NAME}-env'
EXPERIMENT_NAME = f'{PROJECT_NAME}-training'

In [ ]:
# ── IMPORTS ───────────────────────────────────────────────────────────────────
from azure.ai.ml import MLClient
from azure.ai.ml.entities import (
    AmlCompute,
    Environment,
    Data,
)
from azure.ai.ml.constants import AssetTypes
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential

## Section 1 — Connect to Workspace

In [ ]:
# ── AUTH ──────────────────────────────────────────────────────────────────────
try:
    credential = DefaultAzureCredential()
    credential.get_token('https://management.azure.com/.default')
except Exception:
    credential = InteractiveBrowserCredential()

ml_client = MLClient.from_config(credential=credential)

print(f'✓ Connected to workspace : {ml_client.workspace_name}')
print(f'  Subscription           : {ml_client.subscription_id}')
print(f'  Resource group         : {ml_client.resource_group_name}')

## Section 2 — Compute Cluster

In [ ]:
# ── COMPUTE ───────────────────────────────────────────────────────────────────
# Tries to get the cluster; creates it if it doesn't exist.
# min_instances=0 → scales to zero when idle (no idle cost).
try:
    compute = ml_client.compute.get(COMPUTE_NAME)
    print(f'✓ Compute exists: {compute.name}  ({compute.size})')

except Exception:
    compute = AmlCompute(
        name=COMPUTE_NAME,
        size='Standard_DS11_v2',
        min_instances=0,
        max_instances=2,
        idle_time_before_scale_down=120,
    )
    ml_client.compute.begin_create_or_update(compute).result()
    print(f'✓ Compute created: {COMPUTE_NAME}')

## Section 3 — Environment

In [ ]:
# ── ENVIRONMENT ───────────────────────────────────────────────────────────────
# Reads src/conda-env.yml. Azure builds the Docker image once and caches it.
env = Environment(
    name=ENV_NAME,
    conda_file='../src/conda-env.yml',
    image='mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04',
)

env = ml_client.environments.create_or_update(env)
print(f'✓ Environment registered: {env.name}  (version {env.version})')

## Section 4 — Data Asset

In [ ]:
# ── DATA ASSET ────────────────────────────────────────────────────────────────
# Uploads train_clean.csv to the default datastore and registers it.
data_asset = Data(
    name=DATA_ASSET_NAME,
    path='../data/processed/train_clean.csv',
    type=AssetTypes.URI_FILE,
    description=f'Clean training data for {PROJECT_NAME}',
)

data_asset = ml_client.data.create_or_update(data_asset)
print(f'✓ Data asset registered: {data_asset.name}  (version {data_asset.version})')
print(f'  Path: {data_asset.path}')

---
## Phase 2 Gate — verify in Azure Studio before moving to Phase 3

- [ ] **Data tab** → `my-project-data` visible
- [ ] **Compute tab** → `aml-cluster` in Idle state
- [ ] **Environments tab** → `my-project-env` registered

Then update README_TEMPLATE.md → Phase Completion Log → Phase 2 row.